# Entregable 2

## librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score, f1_score)

## Cargar datos y seleccionar variables predictoras

**Target:** `solar_aptittude_class` (Baja, Media, Alta).

**Features (17):** 13 numéricas (`longitude`, `latitude`, `elevation`, `area`, `slope`, `curvature`,
`aspect`, `dist_to_road`, `ambient_temperature`, `ghi`, `humidity`, `wind_speed`, `wind_direction`) y 4
categóricas (`slope_type`, `curvature_type`, `aspect_type`, `dt_wind`), codificadas con one-hot. `area`
tiene 27.69% de valores nulos, imputados con la mediana del conjunto de entrenamiento.

**Excluidas (fuga de información o posteriores a la inversión):**
- `solar_aptitude`, `solar_aptitude_rounded`: definen el target por binning.
- `capacity`, `optimal_tilt`, `pv_potential`: resultado de haber construido y configurado la planta.
- `operational_status`: posterior a la decisión de inversión.
- `OBJECTID`, `code`, `plant_name`: identificadores sin valor predictivo.

**Pendientes de decisión (no incluidas todavía):** `size`, `country`.

In [ ]:
df = pd.read_excel('Dataset_Mundial_Final.xlsx')

NUMERIC_FEATURES = ['longitude', 'latitude', 'elevation', 'area', 'slope', 'curvature', 'aspect',
                     'dist_to_road', 'ambient_temperature', 'ghi', 'humidity', 'wind_speed', 'wind_direction']
CATEGORICAL_FEATURES = ['slope_type', 'curvature_type', 'aspect_type', 'dt_wind']
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = 'solar_aptittude_class'

X = df[FEATURES]
y = df[TARGET]

print(df.shape)
X.head()

## 3. Distribución de clases



In [ ]:
print(y.value_counts())
print()
print((y.value_counts(normalize=True) * 100).round(1).astype(str) + ' %')

plt.figure(figsize=(6, 4))
sns.countplot(x=y, order=['Baja', 'Media', 'Alta'])
plt.title('Distribución de solar_aptittude_class')
plt.ylabel('Número de plantas')
plt.show()

## 4. División train/test estratificada, imputación, codificación y escalado

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocesador = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), NUMERIC_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
])

X_train_s = preprocesador.fit_transform(X_train)
X_test_s = preprocesador.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Dimensión tras imputación/escalado/one-hot:", X_train_s.shape[1])
print("Proporción de clases en train:")
print(y_train.value_counts(normalize=True).round(3))
print("Proporción de clases en test:")
print(y_test.value_counts(normalize=True).round(3))

## 5. LogisticRegression 


In [ ]:
logreg = LogisticRegression(max_iter=2000).fit(X_train_s, y_train)

print("Training set score (accuracy): {:.3f}".format(logreg.score(X_train_s, y_train)))
print("Test set score (accuracy): {:.3f}".format(logreg.score(X_test_s, y_test)))
print()
print(classification_report(y_test, logreg.predict(X_test_s)))

In [ ]:
proba_test_baseline = logreg.predict_proba(X_test_s)
auc_baseline = roc_auc_score(y_test, proba_test_baseline, multi_class='ovr', 
                               average='macro', labels=logreg.classes_)
print("ROC AUC (macro, one-vs-rest), baseline: {:.3f}".format(auc_baseline))

## 6. GridSearchCV

In [ ]:
param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    LogisticRegression(max_iter=2000, class_weight='balanced'),
    param_grid,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1
)
grid.fit(X_train_s, y_train)

print("Mejor C encontrado:", grid.best_params_['C'])
print("Mejor F1 macro (cross-validation):", round(grid.best_score_, 3))

## 7. Evaluación del modelo mejorado (score, error, matriz de confusión, ROC AUC)

In [ ]:
best_model = grid.best_estimator_

print("Training set score (accuracy): {:.3f}".format(best_model.score(X_train_s, y_train)))
print("Test set score (accuracy): {:.3f}".format(best_model.score(X_test_s, y_test)))
print()
print(classification_report(y_test, best_model.predict(X_test_s)))

# ROC AUC multiclase (one-vs-rest, promedio macro)
proba_test = best_model.predict_proba(X_test_s)
auc = roc_auc_score(y_test, proba_test, multi_class='ovr', average='macro', labels=best_model.classes_)
print("ROC AUC (macro, one-vs-rest): {:.3f}".format(auc))

In [ ]:
cm = confusion_matrix(y_test, best_model.predict(X_test_s), labels=['Baja', 'Media', 'Alta'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Baja', 'Media', 'Alta'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Matriz de confusión: LogisticRegression balanceado')
plt.show()

**Interpretación:** ya con las 17 features correctas, el modelo base (sin ajustar) alcanza un recall de
0.57 para `Baja`, muy por encima del recall casi nulo que producía el conjunto de features anterior
(`optimal_tilt` en vez de `slope`/`aspect`/`curvature`). Al incorporar `class_weight='balanced'` y el
mejor `C` encontrado, el recall de `Baja` sube a 0.97, identificando prácticamente todas las plantas de
baja aptitud. El costo es un descenso moderado en accuracy general (de 0.787 a 0.753), muy inferior al que
se observaba antes de corregir las features. El ROC AUC, que no depende del umbral de decisión, se
mantiene idéntico entre ambas versiones (0.886), confirmando que el poder discriminativo real del modelo
no cambia: lo que cambia es el umbral de decisión que la ponderación de clases desplaza a favor de la
clase minoritaria.

## 8. Conclusiones

| Modelo | Train Acc. | Test Acc. | Recall (Baja) | F1 macro | ROC AUC (macro) |
|---|---|---|---|---|---|
| LogisticRegression (default, sin ajustar) | 0.788 | 0.787 | **0.57** | 0.65 | 0.886 |
| LogisticRegression (`class_weight='balanced'`, C óptimo vía GridSearchCV) | 0.760 | 0.753 | **0.97** | 0.62 | 0.886 |

**Corregir el conjunto de features cambió el diagnóstico del modelo.** Con `slope`, `aspect`, `curvature`
y las variables categóricas incluidas, en vez de `optimal_tilt`, el modelo base ya identifica más de la
mitad de las plantas `Baja` (recall 0.57) sin necesidad de balancear clases, muy lejos del recall de 0.01
que producía el conjunto de features anterior.

**El margen de mejora sigue siendo real, aunque más modesto.** Agregando `class_weight='balanced'` y
ajustando `C` vía `GridSearchCV` con `StratifiedKFold`, el recall de `Baja` sube a 0.97 a cambio de una
caída de accuracy de solo 3.4 puntos porcentuales (0.787 a 0.753), un trade-off mucho más favorable que el
observado con las features incorrectas.